# 🐉 Imagen → 3D con Hunyuan3D-2 (gratis, mejor que TripoSR)

Mejor calidad de geometría que TripoSR, en la GPU gratis de Colab. Modo **solo forma** (malla gris), que entra en la T4 (~6 GB).

1. GPU: `Entorno de ejecución` → `Cambiar tipo` → **T4 GPU**.
2. Celda 1 (instalar) → al terminar, **Reiniciar sesión**.
3. Celda 2 (subir imagen) → Celda 3 (generar) → Celda 4 (descargar).

In [ ]:
# Celda 1 — Instalar. Al terminar: Reiniciar sesión.
!nvidia-smi -L
import os
os.chdir('/content')
if not os.path.isdir('/content/Hunyuan3D-2'):
    !git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git
os.chdir('/content/Hunyuan3D-2')
!pip install -q ninja
!pip install -q diffusers transformers accelerate trimesh omegaconf einops opencv-python-headless huggingface_hub
!pip install -q -e . 2>&1 | tail -2
!pip install -q -U 'numpy>=2.1'
print('Instalado. AHORA: Entorno de ejecucion -> Reiniciar sesion, y corre la Celda 2.')

In [ ]:
# Celda 2 — Subir imagen (define IMG)
from google.colab import files
from PIL import Image
up = files.upload()
IMG = list(up.keys())[0]
print('IMG:', IMG, '| modo:', Image.open(IMG).mode)

In [ ]:
# Celda 3 — Generar (la primera vez baja ~10 GB de pesos)
import os, torch
os.chdir('/content/Hunyuan3D-2')
from PIL import Image
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2')
img = Image.open(IMG).convert('RGBA')
mesh = pipe(image=img, num_inference_steps=30, octree_resolution=256, generator=torch.manual_seed(0))[0]
OUT = '/content/hunyuan_mesh.glb'
mesh.export(OUT)
print('Resultado:', ('OK ' + OUT + ' — ' + str(round(os.path.getsize(OUT)/1024, 1)) + ' KB') if os.path.exists(OUT) else 'ERROR')

In [ ]:
# Celda 4 — Descargar
from google.colab import files
files.download('/content/hunyuan_mesh.glb')